# 03. Bài toán người du lịch (TSP) — Genetic Algorithm

Genetic Algorithm mã hóa hoán vị (Order Crossover + Swap Mutation) cho
bài toán người du lịch (TSP). Mỗi cá thể là một **hoán vị** thứ tự
thăm các đỉnh (khác `01`, `02` dùng mã hóa số thực) nên toán tử lai
ghép/đột biến cũng khác hẳn, và không có khái niệm "vi phạm ràng buộc"
(mọi hoán vị đều là lộ trình hợp lệ).

Đối chiếu trên 3 bộ dữ liệu chuẩn từ
[TSPLIB95](https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/tspindex.html)
(`data/data_TSPLIB/`, đều dạng `EDGE_WEIGHT_TYPE=EUC_2D`): berlin52,
rd100, ch150 — kích thước đủ lớn khiến vét cạn bất khả thi
($(n-1)!$ hoán vị), nên dùng **nghiệm tối ưu có sẵn** (file
`.opt.tour`, do TSPLIB công bố) làm mốc so sánh.

Ở quy mô này, GA thuần chỉ với Order Crossover + Swap Mutation chênh
lệch rất lớn so với tối ưu (thử nghiệm: 20-84%) vì thiếu cơ chế "gỡ"
các cạnh cắt nhau trong lộ trình. Nên GA ở đây bật thêm **2-opt local
search** cho một tỉ lệ đủ lớn của quần thể mỗi thế hệ (`two_opt_elite`)
— biến GA thành **memetic algorithm**. Với quần thể 1000 và
`two_opt_elite=50` (5%), cả 3 bộ đều đạt **đúng nghiệm tối ưu tuyệt
đối** chỉ sau 50 thế hệ. Lưu ý: `two_opt_elite` phải theo TỈ LỆ với
quần thể — thử với số cố định nhỏ (2 cá thể) trên quần thể lớn cho kết
quả tệ hơn cả quần thể nhỏ, vì cá thể đã tinh chỉnh quá hiếm để lan
gen ra qua chọn lọc giải đấu.

| Bộ dữ liệu | Số đỉnh | Nghiệm tối ưu |
|---|---|---|
| berlin52 | 52 | 7542 |
| rd100 | 100 | 7910 |
| ch150 | 150 | 6528 |


In [1]:
import itertools
import random
import time

import numpy as np
from IPython.display import Markdown, display


def tour_length(tour, distance_matrix):
    """Tổng trọng số của lộ trình khép kín (quay lại đỉnh xuất phát)."""

    tour = np.asarray(tour)

    tiep_theo = np.roll(tour, -1)

    return float(distance_matrix[tour, tiep_theo].sum())


# ============================================================
# GENETIC ALGORITHM (MÃ HÓA HOÁN VỊ)
# ============================================================

# Tham số GA dùng chung, cùng phong cách với 01/02.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3


def genetic_algorithm_tsp(
    distance_matrix,

    population_size=200,
    generations=300,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,
    two_opt_elite=0,

    seed=42,
):
    """
    GA mã hóa hoán vị cho bài toán người du lịch (TSP).

    Khác GA số thực ở 01/02: cá thể là một HOÁN VỊ của [0..n-1] (thứ tự
    thăm các đỉnh), không phải một điểm trong không gian số thực - nên
    dùng lai ghép/đột biến riêng cho hoán vị (Order Crossover, Swap
    Mutation) thay vì blend crossover / Gaussian mutation. Không có
    khái niệm "vi phạm ràng buộc": mọi hoán vị đều là lộ trình hợp lệ.

    Cài bằng list/`random` thuần Python thay vì mảng/`Generator` numpy
    cho từng cá thể: n thường nhỏ (vài chục đỉnh), mà số lần gọi hàm
    random/lập chỉ số lại rất nhiều (mỗi lần lai ghép, đột biến) - với
    thao tác nhỏ như vậy, overhead cố định của numpy trên mỗi lần gọi
    lớn hơn hẳn phần tính toán thật, làm GA chậm đi cả chục lần.

    `two_opt_elite` (mặc định 0 = tắt): số cá thể tốt nhất mỗi thế hệ
    được áp thêm 2-opt local search - biến GA thành MEMETIC ALGORITHM.
    OX + Swap thuần không có cơ chế "gỡ" các cạnh cắt nhau trong lộ
    trình nên rất dễ mắc kẹt xa nghiệm tối ưu khi n lớn (xem bộ dữ liệu
    TSPLIB bên dưới); bật 2-opt cho vài cá thể tốt nhất mỗi thế hệ cải
    thiện gap rất nhiều với chi phí thấp, không cần áp cho cả quần thể.
    """

    n = len(distance_matrix)

    D = distance_matrix.tolist()  # index bằng list Python nhanh hơn mảng numpy ở đây

    py_rng = random.Random(seed)

    def do_dai(tour):
        tong = D[tour[-1]][tour[0]]
        for i in range(n - 1):
            tong += D[tour[i]][tour[i + 1]]
        return tong

    # --------------------------------------------------------
    # Initial population: mỗi cá thể là một hoán vị ngẫu nhiên
    # --------------------------------------------------------

    population = []
    for _ in range(population_size):
        ca_the = list(range(n))
        py_rng.shuffle(ca_the)
        population.append(ca_the)

    def evaluate(pop):
        return [do_dai(tour) for tour in pop]

    # --------------------------------------------------------
    # Tournament selection
    # --------------------------------------------------------

    def tournament_selection(fitness):

        best_index = py_rng.randrange(population_size)
        best_value = fitness[best_index]

        for _ in range(tournament_size - 1):
            i = py_rng.randrange(population_size)
            if fitness[i] < best_value:
                best_index, best_value = i, fitness[i]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Order Crossover (OX)
    # --------------------------------------------------------
    #
    # Sao chép nguyên một đoạn liên tiếp của cha vào con, phần còn lại
    # điền theo ĐÚNG thứ tự xuất hiện ở mẹ (bỏ qua đỉnh đã có) - luôn
    # cho ra một hoán vị hợp lệ, không lặp/thiếu đỉnh nào.

    def order_crossover(parent1, parent2):

        if py_rng.random() > crossover_rate:
            return parent1.copy(), parent2.copy()

        dau, cuoi = sorted(py_rng.sample(range(n), 2))

        def lai(cha, me):

            da_co = set(cha[dau:cuoi + 1])
            con_lai = iter(dinh for dinh in me if dinh not in da_co)

            con = [None] * n
            con[dau:cuoi + 1] = cha[dau:cuoi + 1]

            for vi_tri in itertools.chain(range(dau), range(cuoi + 1, n)):
                con[vi_tri] = next(con_lai)

            return con

        return lai(parent1, parent2), lai(parent2, parent1)

    # --------------------------------------------------------
    # Swap Mutation
    # --------------------------------------------------------
    #
    # mutation_rate ở đây là xác suất CẢ CÁ THỂ chịu một lần đột biến
    # (hoán đổi 2 vị trí ngẫu nhiên) - khác quy ước "per-gene" ở GA số
    # thực, vì hoán vị không có khái niệm đột biến độc lập từng gene
    # (đổi 1 gene mà không đổi gene khác thì phá vỡ tính hoán vị).

    def swap_mutation(tour):

        if py_rng.random() < mutation_rate:
            i, j = py_rng.sample(range(n), 2)
            tour[i], tour[j] = tour[j], tour[i]

        return tour

    # --------------------------------------------------------
    # 2-opt (tùy chọn - memetic algorithm)
    # --------------------------------------------------------
    #
    # Xét mọi cặp cạnh (i,i+1) và (j,j+1) không kề nhau, thử đảo ngược
    # đoạn giữa hai cạnh để "gỡ" đường chéo nhau - giữ lại nếu tổng độ
    # dài giảm. Lặp lại đến khi một lượt quét không cải thiện được gì
    # nữa (hội tụ về một cực trị cục bộ theo lân cận 2-opt).

    def two_opt(tour, max_passes=200):

        tour = tour.copy()

        for _ in range(max_passes):

            improved = False

            for i in range(n - 1):
                a, b = tour[i], tour[i + 1]
                for j in range(i + 2, n):
                    if i == 0 and j == n - 1:
                        continue
                    c, d = tour[j], tour[(j + 1) % n]
                    if D[a][c] + D[b][d] < D[a][b] + D[c][d] - 1e-9:
                        tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                        b = tour[i + 1]
                        improved = True

            if not improved:
                break

        return tour

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []

    start_time = time.perf_counter()

    fitness = evaluate(population)

    best_index = min(range(population_size), key=lambda i: fitness[i])
    best_tour = population[best_index].copy()
    best_length = fitness[best_index]

    for generation in range(generations):

        if two_opt_elite > 0:
            order_2opt = sorted(range(population_size), key=lambda i: fitness[i])
            for i in order_2opt[:two_opt_elite]:
                population[i] = two_opt(population[i])
            fitness = evaluate(population)

        current_best = min(range(population_size), key=lambda i: fitness[i])
        if fitness[current_best] < best_length:
            best_length = fitness[current_best]
            best_tour = population[current_best].copy()

        history.append(best_length)

        # Elitism: giữ nguyên elite_size lộ trình tốt nhất
        order = sorted(range(population_size), key=lambda i: fitness[i])
        new_population = [population[i].copy() for i in order[:elite_size]]

        while len(new_population) < population_size:

            parent1 = tournament_selection(fitness)
            parent2 = tournament_selection(fitness)

            child1, child2 = order_crossover(parent1, parent2)

            new_population.append(swap_mutation(child1))

            if len(new_population) < population_size:
                new_population.append(swap_mutation(child2))

        population = new_population
        fitness = evaluate(population)

    current_best = min(range(population_size), key=lambda i: fitness[i])
    if fitness[current_best] < best_length:
        best_length = fitness[current_best]
        best_tour = population[current_best].copy()

    elapsed_time = time.perf_counter() - start_time

    generations_run = next(
        (g + 1 for g, value in enumerate(history) if value == best_length),
        generations,
    )

    return {
        "tour": np.array(best_tour),
        "length": float(best_length),
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "generations_run": generations_run,
        "seed": seed,
    }


def _num(value, digits=6):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


In [2]:
import gzip
from pathlib import Path


def read_tsplib_tsp(path):
    """
    Đọc file .tsp(.gz) dạng NODE_COORD_SECTION + EDGE_WEIGHT_TYPE=EUC_2D,
    trả về ma trận khoảng cách. Chỉ số đỉnh trả về là 0-indexed (khớp
    với genetic_algorithm_tsp/tour_length ở trên).
    """

    coords = {}
    edge_weight_type = None
    section = None

    with gzip.open(path, "rt") as f:
        for dong in f:
            dong = dong.strip()
            if not dong or dong == "EOF":
                continue
            if dong.upper().startswith("EDGE_WEIGHT_TYPE"):
                edge_weight_type = dong.split(":", 1)[1].strip()
            if dong == "NODE_COORD_SECTION":
                section = "coord"
                continue
            if section == "coord":
                chi_so, x, y = dong.split()
                coords[int(chi_so)] = (float(x), float(y))

    if edge_weight_type != "EUC_2D":
        raise ValueError(
            f"Chỉ hỗ trợ EDGE_WEIGHT_TYPE=EUC_2D, file này là {edge_weight_type!r}."
        )

    n = len(coords)
    toa_do = np.array([coords[i + 1] for i in range(n)])

    # EUC_2D: khoảng cách Euclid, làm tròn về số nguyên gần nhất (chuẩn TSPLIB).
    hieu = toa_do[:, None, :] - toa_do[None, :, :]
    distance_matrix = np.round(np.sqrt((hieu ** 2).sum(axis=-1)))

    return distance_matrix


def read_tsplib_opt_tour(path):
    """Đọc file .opt.tour(.gz), trả về lộ trình tối ưu (0-indexed)."""

    tour = []
    section = None

    with gzip.open(path, "rt") as f:
        for dong in f:
            dong = dong.strip()
            if not dong:
                continue
            if dong == "TOUR_SECTION":
                section = "tour"
                continue
            if section == "tour":
                if dong in ("-1", "EOF"):
                    break
                tour.extend(int(v) - 1 for v in dong.split())

    return np.array(tour)


In [3]:
TSPLIB_DIR = Path("data/data_TSPLIB")

# Bật 2-opt cho một PHẦN TRĂM đủ lớn của quần thể mỗi thế hệ (memetic
# algorithm) - nếu chỉ dùng OX + Swap thuần (không 2-opt) thì chênh
# lệch với tối ưu rất lớn khi n lớn (đã thử: 20-84%). two_opt_elite
# phải theo TỈ LỆ với population_size chứ không phải số cố định: thử
# two_opt_elite=2 trên quần thể 1000 (chỉ 0.2%) cho kết quả TỆ HƠN cả
# quần thể nhỏ, vì cá thể đã tinh chỉnh quá hiếm để lan gen ra cả quần
# thể qua chọn lọc giải đấu. Với two_opt_elite=50 (5%) trên quần thể
# 1000, chỉ cần 50 thế hệ là cả 3 bộ đều đạt ĐÚNG nghiệm tối ưu tuyệt
# đối (0% chênh lệch) - dù rd100 khá sát ngưỡng (đạt ở thế hệ 48/50).
# Đã thử tăng two_opt_elite lên 100 (10%): không cải thiện gì thêm (đã
# 0% chênh lệch rồi) mà chạy chậm hơn hẳn - 50 là điểm đủ dùng.
TSPLIB_INSTANCES = [
    ("berlin52", 50),
    ("rd100", 50),
    ("ch150", 50),
]

tsplib_results = []

for ten, so_the_he in TSPLIB_INSTANCES:

    distance_matrix = read_tsplib_tsp(TSPLIB_DIR / f"{ten}.tsp.gz")
    optimal_tour = read_tsplib_opt_tour(TSPLIB_DIR / f"{ten}.opt.tour.gz")
    optimal_length = tour_length(optimal_tour, distance_matrix)

    ga_result = genetic_algorithm_tsp(
        distance_matrix,
        population_size=1000,
        generations=so_the_he,
        two_opt_elite=50,
        seed=42,
    )

    tsplib_results.append({
        "ten": ten,
        "n": len(distance_matrix),
        "optimal": optimal_length,
        "ga_length": ga_result["length"],
        "ga_time": ga_result["time"],
        "generations_run": ga_result["generations_run"],
        "generations": so_the_he,
    })

    print(
        f"{ten} (n={len(distance_matrix)}): GA={ga_result['length']:.0f} "
        f"| tối ưu={optimal_length:.0f} "
        f"| đạt ở thế hệ {ga_result['generations_run']}/{so_the_he} "
        f"| {ga_result['time']:.1f}s"
    )


berlin52 (n=52): GA=7542 | tối ưu=7542 | đạt ở thế hệ 8/50 | 6.3s


rd100 (n=100): GA=7910 | tối ưu=7910 | đạt ở thế hệ 48/50 | 30.2s


ch150 (n=150): GA=6528 | tối ưu=6528 | đạt ở thế hệ 37/50 | 70.6s


In [4]:
def show_tsplib_comparison(rows):
    """So sánh GA với nghiệm tối ưu TSPLIB cho nhiều bộ dữ liệu."""

    dong = [
        "| Bộ dữ liệu | Số đỉnh | Tối ưu | GA | Đạt tối ưu ở thế hệ | Thời gian GA (s) |",
        "|---|---|---|---|---|---|",
    ]

    for r in rows:
        dong.append(
            "| " + r["ten"] + " | " + str(r["n"]) + " | $" + _num(r["optimal"])
            + "$ | $" + _num(r["ga_length"]) + "$ | " + str(r["generations_run"])
            + " / " + str(r["generations"])
            + " | $" + _num(r["ga_time"], 4) + "$ |"
        )

    display(Markdown("\n".join(dong)))


show_tsplib_comparison(tsplib_results)


| Bộ dữ liệu | Số đỉnh | Tối ưu | GA | Đạt tối ưu ở thế hệ | Thời gian GA (s) |
|---|---|---|---|---|---|
| berlin52 | 52 | $7542.000000$ | $7542.000000$ | 8 / 50 | $6.2922$ |
| rd100 | 100 | $7910.000000$ | $7910.000000$ | 48 / 50 | $30.2455$ |
| ch150 | 150 | $6528.000000$ | $6528.000000$ | 37 / 50 | $70.6206$ |